In [401]:
"""
%pip install dwave-ocean-sdk
"""

'\n%pip install dwave-ocean-sdk\n'

In [402]:
import dimod
from dwave.samplers import SimulatedAnnealingSampler

# Exact Cover (EC) as a 2-SAT Binary Quadratic Model (BQM) for Quantum Annealing

In this experiment, we explore the Exact Cover (EC) problem, a combinatorial optimization problem in which we must select subsets from a given collection such that every element of the original set is covered exactly once, without repetitions.


## 1.1 Exact Cover Problem


Given a universe \(U\), a collection of subsets (\S_1,\...,\S_m), and binary variables (x_i in {0,1\}) indicating whether subset \(S_i\) is selected, the Exact Cover problem can be formulated as:

$$
\boxed{\sum_{i:u\in S_i}x_i=1,\quad \forall u\in U}
$$

This notebook focuses exclusively on a 2-SAT-based formulation of Exact Cover, considering a simplified instance in which each element belongs to exactly two subsets. Each restriction can therefore be expressed as an *exactly-one* condition, equivalent to two CNF clauses:

$$
(x_i\lor x_j)\land(\neg x_i\lor\neg x_j)
$$

To formulate these restrictions as an optimization problem, we assign an energy penalty to each violated condition:

$$
f_c(x_i,x_j)=(1-x_i-x_j)^2
$$

By summing the penalties, we obtain a Binary Quadratic Model (BQM), whose general form is:

$$
\boxed{f(x)=c+\sum_i a_ix_i+\sum_{i<j}b_{ij}x_ix_j}
$$

For our specific Exact Cover instance, the restrictions involve the pairs \((1,4)\), \((1,3)\), \((2,3)\), and \((2,4)\). Consequently, our BQM becomes:

$$
\boxed{
f_{EC}(x)=4-2\sum_{i=1}^{4}x_i+
2(x_1x_4+x_1x_3+x_2x_3+x_2x_4)
}
$$

The optimal solutions correspond to configurations with minimum energy equal to zero, meaning all Exact Cover restrictions are satisfied.

## 1.2 Ocean SDK

Using D-Wave's Ocean SDK, we will construct this BQM, validate its optimal solutions using an exact solver, and investigate its behavior using classical simulated annealing. The same model can subsequently be submitted to a quantum annealer.

Quantum annealers typically implement an Ising Hamiltonian, expressed in terms of spin variables \(s_i\in\{-1,+1\}\). The conversion between binary and spin variables is defined by:

$$
\boxed{x_i=\frac{1-s_i}{2}}
$$

Thus, \(x_i=0\) corresponds to \(s_i=+1\), while \(x_i=1\) corresponds to \(s_i=-1\).

Under this transformation, our objective function becomes:

$$
E(s)=2+\frac12(s_1s_4+s_1s_3+s_2s_3+s_2s_4)
$$

In the quantum formulation, spin variables are replaced by Pauli-Z operators, producing the final Hamiltonian:

$$
\boxed{
H_F=2I+\frac12(Z_1Z_4+Z_1Z_3+Z_2Z_3+Z_2Z_4)
}
$$

The eigenvalues of this Hamiltonian correspond to the energies of the classical configurations, and its ground states represent the optimal solutions.


## Note:
For this experiment, however, we only need to construct the binary BQM. The Ocean SDK handles the necessary conversion from binary variables to the Ising representation, allowing us to focus on problem formulation, optimization, and result analysis.

# 1°: Build Fundamental Functions

In [403]:
def define_restrictions(subsets: list)->list:
  """
  This function scans all possible subsets and searches
  for duplicate elements, considering the simultaneous
  intersection of any two of them.
  """
  n_subsets = len(subsets)
  restrictions = []

  for i in range(n_subsets):
    for j in range(i + 1, n_subsets):
      if subsets[i] & subsets[j]:
        restrictions.append((i + 1, j + 1))

  # Return the restrictions and the number of clauses
  return restrictions, len(restrictions)

In [404]:
def linear_terms(restrictions: list) -> dict:
    """
    Extract linear coefficients from pairwise exactly-one restrictions.
    Each variable receives -1 for every restriction in which it appears.
    """
    linear = {}

    for a, b in restrictions:
        for variable in (a, b):
            term = f"x{variable}"
            linear[term] = linear.get(term, 0) - 1

    return linear

In [405]:
def quadratic_terms(restrictions: list) -> dict:
    """
    Extract quadratic coefficients from pairwise exactly-one restrictions.
    Each pair contributes +2 to its quadratic coefficient.
    """
    quadratic = {}

    for a, b in restrictions:
        pair = (f"x{min(a, b)}", f"x{max(a, b)}")
        quadratic[pair] = quadratic.get(pair, 0) + 2

    return quadratic

In [406]:
def two_sat_bqm(restrictions:list, offset: float)->list:
  """
  Given the restrictions, we should be able to build
  the 2-SAT problem as a Binary Quadratic Model corresponding
  through the following formula:

  f(x) = m - SUM B_i * x_i + 2 SUM C_ij * x_i * x_j
  """

  linear = linear_terms(restrictions)
  quadratic = quadratic_terms(restrictions)

  bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset, dimod.BINARY)
  return bqm

In [407]:
def evaluate_results(original_set: set, subsets: list, bit_list: list) -> None:
    result = set()
    intersection = set()

    for i in range(len(bit_list)):
        if bit_list[i] == 1:
            intersection |= subsets[i] & result
            result |= subsets[i]

    if result == original_set and len(intersection) == 0:
        print(f"The bitstring {bit_list} is a valid Exact Cover solution")
        print(f"Original set: {original_set}")
        print(f"Computed set: {result}")
        print(f"Intersection: {intersection}")

    elif result == original_set and len(intersection) > 0:
        print(f"The bitstring {bit_list} covers the original set")
        print("but contains duplicated elements")
        print(f"Original set: {original_set}")
        print(f"Computed set: {result}")
        print(f"Intersection: {intersection}")

    else:
        print(f"The bitstring {bit_list} is not a valid Exact Cover solution")
        print(f"Original set: {original_set}")
        print(f"Computed set: {result}")
        print(f"Intersection: {intersection}")

# 2°: Solve the Exact Cover modeled as a BQM in a Quantum Annealer

In [408]:
# Original set, or the set we wish to obtain by combining its subsets
original_set = {1,2,3,4}

# Subsets of the original set
set1 = {1,2}
set2 = {3,4}
set3 = {2,3}
set4 = {1,4}

# List of the subsets
subsets = [set1, set2, set3, set4]

# Here we build the restrictions and the recover the offset factor
restrictions, offset_factor =  define_restrictions(subsets)

In [409]:
# Build the binary quadratic model considering the 2-SAT formulation
bqm = two_sat_bqm(restrictions, offset_factor)

In [410]:
# Here we compute the exact solution for out problem
exact = dimod.ExactSolver()
print("Exact Solutions:")
print(exact.sample(bqm).lowest())

Exact Solutions:
  x1 x2 x3 x4 energy num_oc.
0  0  0  1  1    0.0       1
1  1  1  0  0    0.0       1
['BINARY', 2 rows, 2 samples, 4 variables]


In [411]:
# Here we compute the solution considering the simulation of a
# Quantum Annealer
sampler = SimulatedAnnealingSampler()
result = sampler.sample(bqm, num_reads=1024)
result.aggregate().to_pandas_dataframe()

,x1,x2,x3,x4,energy,num_occurrences
0,0,1,0,1,2.0,59
1,1,1,0,0,0.0,397
2,0,0,1,1,0.0,378
3,1,0,1,0,2.0,65
4,0,1,1,0,2.0,54
5,1,0,0,1,2.0,71


# 3°: Validate the results find in the Quantum Annealer

In [412]:
# Extract the lowest energy bitstrings
ground_states = result.lowest().aggregate()

# Convert the bitstrings to a list of bits
bit_lists = [
    [int(sample[v]) for v in ground_states.variables]
    for sample in ground_states.samples()
]

# List of all possible results as bit lists
bit_lists

[[1, 1, 0, 0], [0, 0, 1, 1]]

In [413]:
# Validate the lowest eingenstates found
for bit_list in bit_lists:
  evaluate_results(original_set, subsets, bit_list)
  print("\n")

The bitstring [1, 1, 0, 0] is a valid Exact Cover solution
Original set: {1, 2, 3, 4}
Computed set: {1, 2, 3, 4}
Intersection: set()


The bitstring [0, 0, 1, 1] is a valid Exact Cover solution
Original set: {1, 2, 3, 4}
Computed set: {1, 2, 3, 4}
Intersection: set()


